In [0]:
--  set catalog + schema
USE CATALOG ymotorna_DB;
USE SCHEMA superstore_lakehouse;

-- check
SELECT current_catalog(), 
    current_schema(); 

current_catalog(),current_schema()
ymotorna_db,superstore_lakehouse


In [0]:
show tables

database,tableName,isTemporary
superstore_lakehouse,bronze_orders,false
superstore_lakehouse,gold_customer_metrics,false
superstore_lakehouse,gold_sales_category,false
superstore_lakehouse,gold_sales_daily,false
superstore_lakehouse,gold_sales_region,false
superstore_lakehouse,products,false
superstore_lakehouse,silver_orders,false
superstore_lakehouse,silver_rejected_orders,false
,silver_cleaned,true


### Create Silver layer

In [0]:
-- +view w/ cleaned data  \\ reuse later
-- lowercase_trim  \\  dtype \\ filter
create or replace temp view silver_cleaned as
with deduplicated as (
    select *,
        row_number() over (partition by row_id
                          order by ingestion_timestamp asc) as row_num
    from bronze_orders
),

cleaned as  (
    select
        row_id,
        order_id,
        try_cast(order_date as date) as order_date,
        try_cast(ship_date as date) as ship_date,                          
        lower(trim(ship_mode)) as ship_mode,
        customer_id,
        trim(customer_name) as customer_name,
        lower(trim(segment)) as segment,
        trim(country) as country,
        trim(city) as city,
        trim(state) as state,
        postal_code,
        lower(trim(region)) as region,
        product_id,
        lower(trim(category)) as category,
        lower(trim(sub_category)) as sub_category,
        trim(product_name) as product_name,
        try_cast(sales as double) as sales,
        try_cast(quantity as integer) as quantity,
        try_cast(discount as double) as discount,
        profit,
        ingestion_timestamp,
        source_file_name
    from deduplicated
    where row_num=1)

select *,
    (case 
        when row_id is null then 'null row_id'
        when order_id is null then 'null order_id'
        when order_date is null then 'null or unparseable order_date'
        when ship_date is null then 'null or unparseable ship_date'
        when ship_date < order_date then 'ship_date before order_date'
        when customer_id is null then 'null customer_id'
        when customer_name is null then 'null customer_name'
        when segment is null then 'null segment'
        when country is null then 'null country'
        when city is null then 'null city'
        when state is null then 'null state'
        when region is null then 'null region'
        when product_id is null then 'null product_id'
        when category is null then 'null category'
        when product_name is null then 'null product_name'
        when sales is null then 'null sales'
        when sales < 0 then 'sales < 0'
        when quantity is null then 'null quantity'
        when quantity <= 0 then 'quantity <= 0'
        when discount is null then 'null discount'
        when discount < 0 then 'discount < 0'
        else null end) as rejection_reason
from cleaned;


-- drop table if exists silver_orders;
-- drop table if exists silver_rejected_orders;

-- +silver tbl
create table if not exists silver_orders as
select row_id, order_id, order_date, ship_date, ship_mode, customer_id, customer_name, segment, country, city, state, postal_code, region, product_id, category, sub_category,  
        product_name, sales, quantity, discount, profit, ingestion_timestamp, source_file_name
from silver_cleaned
where rejection_reason is null;  


-- +regected tbl
create table if not exists silver_rejected_orders as
select * 
from silver_cleaned 
where rejection_reason is not null;


num_affected_rows,num_inserted_rows


---
###Add incremental logic

In [0]:
-- merge
merge into silver_orders as target
using (
    select row_id, order_id, order_date, ship_date, ship_mode, customer_id, customer_name, segment, country, city, state, postal_code, region, product_id, category, sub_category,  
        product_name, sales, quantity, discount, profit, ingestion_timestamp, source_file_name
    from silver_cleaned
    where ingestion_timestamp > (select max(ingestion_timestamp) from silver_orders)
        and rejection_reason is null
) as source
on target.row_id = source.row_id
when matched then update set *
when not matched then insert *;


merge into silver_rejected_orders as target
using (
    select row_id, order_id, order_date, ship_date, ship_mode, customer_id, customer_name, segment, country, city, state, postal_code, region, product_id, category, sub_category,  
        product_name, sales, quantity, discount, profit, ingestion_timestamp, source_file_name, rejection_reason
    from silver_cleaned
    where ingestion_timestamp > (select max(ingestion_timestamp) from silver_orders)
        and rejection_reason is not null
) as source
on target.row_id = source.row_id
when matched then update set *
when not matched then insert *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


Check

In [0]:
select 'silver_orders' as tbl, 
    count(*) as rows from silver_orders
union all
select 'silver_rejected_orders' as tbl, 
    count(*) as rows from silver_rejected_orders;


tbl,rows
silver_orders,9696
silver_rejected_orders,306


In [0]:
describe history silver_orders;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
5,2026-06-29T15:52:39.000Z,73808821626549,ymotorna@kse.org.ua,MERGE,"Map(predicate -> [""(row_id#81695 = row_id#81577)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(1817384817085710),ad8bc33d-bb70-4b7a-a0c9-9f0d2c7ede7d,0629-101017-f0mg8rm1-v2n,4,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 6672, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 2802, materializeSourceTimeMs -> 665, numTargetRowsInserted -> 2, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 884, numTargetRowsUpdated -> 0, numOutputRows -> 2, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1218)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
4,2026-06-29T15:51:28.000Z,73808821626549,ymotorna@kse.org.ua,MERGE,"Map(predicate -> [""(row_id#77347 = row_id#77229)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(1817384817085710),003e248d-f562-44b8-b65c-f9f3b2a6d813,0629-101017-f0mg8rm1-v2n,3,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 0, numTargetBytesAdded -> 0, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 2023, materializeSourceTimeMs -> 691, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 482, numTargetRowsUpdated -> 0, numOutputRows -> 0, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 0, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 806)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
3,2026-06-29T15:50:01.000Z,73808821626549,ymotorna@kse.org.ua,MERGE,"Map(predicate -> [""(row_id#73488 = row_id#73370)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(1817384817085710),b003ff96-2c00-4b8b-a617-aa82a882cd23,0629-101017-f0mg8rm1-v2n,2,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 0, numTargetBytesAdded -> 0, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 1955, materializeSourceTimeMs -> 591, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 578, numTargetRowsUpdated -> 0, numOutputRows -> 0, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 0, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 757)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
2,2026-06-29T15:47:59.000Z,73808821626549,ymotorna@kse.org.ua,MERGE,"Map(predicate -> [""(row_id#72396 = row_id#72278)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(1817384817085710),b8a9c84e-149c-4b83-a403-b85124350ddd,0629-101017-f

In [0]:
describe history silver_rejected_orders;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2026-06-29T15:52:42.000Z,73808821626549,ymotorna@kse.org.ua,MERGE,"Map(predicate -> [""(row_id#82587 = row_id#82491)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(1817384817085710),03901aa3-29bd-43f7-811b-ba51a0acccf6,0629-101017-f0mg8rm1-v2n,2,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 0, numTargetBytesAdded -> 0, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 2047, materializeSourceTimeMs -> 699, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 548, numTargetRowsUpdated -> 0, numOutputRows -> 0, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 0, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 769)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
2,2026-06-29T15:51:32.000Z,73808821626549,ymotorna@kse.org.ua,MERGE,"Map(predicate -> [""(row_id#78215 = row_id#78119)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(1817384817085710),721ce01f-bdc3-413d-ba7c-9a9097ff1f0b,0629-101017-f0mg8rm1-v2n,1,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 7822, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 2581, materializeSourceTimeMs -> 709, numTargetRowsInserted -> 6, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 882, numTargetRowsUpdated -> 0, numOutputRows -> 6, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 6, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 961)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
1,2026-06-29T15:50:04.000Z,73808821626549,ymotorna@kse.org.ua,MERGE,"Map(predicate -> [""(row_id#74390 = row_id#74271)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(1817384817085710),de942c84-9fe0-4f72-9928-a1326ccc2e18,0629-101017-f0mg8rm1-v2n,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 0, numTargetBytesAdded -> 0, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 1918, materializeSourceTimeMs -> 654, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 598, numTargetRowsUpdated -> 0, numOutputRows -> 0, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 0, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 631)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
0,2026-06-29T15:47:27.000Z,73808821626549,ymotorna@kse.org.ua,CREATE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> 

In [0]:
select * from silver_rejected_orders

row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,state,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit,ingestion_timestamp,source_file_name,rejection_reason
33,US-2015-150630,2015-09-17,2015-09-21,standard class,TB-21520,Tracy Blumstein,consumer,United States,Philadelphia,Pennsylvania,19140,east,OFF-BI-10001525,office supplies,binders,"""Acco Pressboard Covers with Storage Hooks, 14 7/8"""" x 11""""",null,null,6.0,0.7,2026-06-29T15:43:29.106Z,dbfs:/Volumes/ymotorna_db/superstore_lakehouse/raw_data/initial/superstore_data.csv,null sales
77,US-2017-118038,2017-12-09,2017-12-11,first class,KB-16600,Ken Brennan,corporate,United States,Houston,Texas,77041,central,FUR-FU-10000260,furniture,furnishings,"""6"""" Cubicle Wall Clock",null,null,3.0,0.6,2026-06-29T15:43:29.106Z,dbfs:/Volumes/ymotorna_db/superstore_lakehouse/raw_data/initial/superstore_data.csv,null sales
106,US-2015-156867,2015-11-13,2015-11-17,standard class,LC-16870,Lena Cacioppo,consumer,United States,Aurora,Colorado,80013,west,OFF-BI-10002794,office supplies,binders,"""Avery Trapezoid Ring Binder, 3"""" Capacity",null,null,36.882,3.0,2026-06-29T15:43:29.106Z,dbfs:/Volumes/ymotorna_db/superstore_lakehouse/raw_data/initial/superstore_data.csv,null sales
199,US-2017-124303,2017-07-06,2017-07-13,standard class,FH-14365,Fred Hopkins,corporate,United States,Philadelphia,Pennsylvania,19120,east,OFF-BI-10000343,office supplies,binders,"""Pressboard Covers with Storage Hooks, 9 1/2"""" x 11""""",null,null,2.0,0.7,2026-06-29T15:43:29.106Z,dbfs:/Volumes/ymotorna_db/superstore_lakehouse/raw_data/initial/superstore_data.csv,null sales
256,US-2015-159982,2015-11-28,2015-12-04,standard class,DR-12880,Dan Reichenbach,corporate,United States,Chicago,Illinois,60623,central,OFF-ST-10004804,office supplies,storage,"""Belkin 19"""" Vented Equipment Shelf",null,null,2.0,0.2,2026-06-29T15:43:29.106Z,dbfs:/Volumes/ymotorna_db/superstore_lakehouse/raw_data/initial/superstore_data.csv,null sales
275,CA-2017-118136,2017-09-16,2017-09-17,first class,BB-10990,Barry Blumstein,corporate,United States,Inglewood,California,90301,west,OFF-PA-10002615,office supplies,paper,"""Ampad Gold Fibre Wirebound Steno Books, 6"""" x 9""""",null,null,2.0,0.0,2026-06-29T15:43:29.106Z,dbfs:/Volumes/ymotorna_db/superstore_lakehouse/raw_data/initial/superstore_data.csv,null sales
278,CA-2017-132976,2017-10-13,2017-10-17,standard class,AG-10495,Andrew Gjertsen,corporate,United States,Philadelphia,Pennsylvania,19140,east,OFF-PA-10004470,office supplies,paper,"""Adams Write n' Stick Phone Message Book, 11"""" X 5 1/4""""",null,null,4.0,0.2,2026-06-29T15:43:29.106Z,dbfs:/Volumes/ymotorna_db/superstore_lakehouse/raw_data/initial/superstore_data.csv,null sales
300,CA-2016-142545,2016-10-28,2016-11-03,standard class,JD-15895,Jonathan Doherty,corporate,United States,Belleville,New Jersey,7109,east,OFF-ST-10002756,office supplies,storage,"""Tennsco Stur-D-Stor Boltless Shelving, 5 Shelves, 24"""" Deep",null,null,8.0,0.0,2026-06-29T15:43:29.106Z,dbfs:/Volumes/ymotorna_db/superstore_lakehouse/raw_data/initial/superstore_data.csv,null sales
340,CA-2015-128167,2015-06-22,2015-06-26,second class,KL-16645,Ken Lonsdale,consumer,United States,Layton,Utah,84041,west,OFF-FA-10000490,office supplies,fasteners,"""OIC Binder Clips, Mini, 1/4"""" Capacity",null,null,4.0,0.0,2026-06-29T15:43:29.106Z,dbfs:/Volumes/ymotorna_db/superstore_lakehouse/raw_data/initial/superstore_data.csv,null sales
344,CA-2014-122336,2014-04-13,2014-04-17,second class,JD-15895,Jonathan Doherty,corporate,United States,Philadelphia,Pennsylvania,19140,east,TEC-PH-10000702,technology,phones,"""Square Credit Card Reader, 4 1/2"""" x 4 1/2"""" x 1""""",null,null,12.0,0.4,2026-06-29T15:43:29.106Z,dbfs:/Volumes/ymotorna_db/superstore_lakehouse/raw_data/initial/superstore_data.csv,null sales
